In [1]:
from dotenv import load_dotenv
import os

project_name = 'bigquery-public-data'
dataset_name = 'san_francisco_311'
table_name = '311_service_requests'  

# Make sure to go to your service account and add BigQuery User role
load_dotenv()
credential_path = os.getenv("GCP_SERVICE_ACCOUNT_KEY") 
gcp_project_id = os.getenv("GCP_PROJECT_ID")

In [2]:
# Creating an engine
from sqlalchemy import create_engine
from sqlalchemy import MetaData, Table
from sqlalchemy import select, and_

engine = create_engine(f"bigquery://{gcp_project_id}",
                       credentials_path=credential_path)

In [3]:
metadata = MetaData()
sf_table = Table(
        table_name,
        metadata, # added empty MetaData object
        autoload_with=engine,
        schema=f"{project_name}.{dataset_name}")

/Users/dwoodbridge/Class/2026_MSDS692/.venv/lib/python3.13/site-packages/google/cloud/bigquery/client.py:626: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


In [4]:
metadata.reflect(bind=engine)

In [5]:
from sqlalchemy import func
# Return the status, police_district, and count of unique_key, grouped by status and police_district, ordered by count descending
query = select(sf_table.c.status, sf_table.c.police_district, func.count(sf_table.c.unique_key))\
        .group_by(sf_table.c.status, sf_table.c.police_district)\
        .order_by(func.count(sf_table.c.unique_key).desc())
with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(row)

('Closed', 'MISSION', 1520922)
('Closed', 'NORTHERN', 1093485)
('Closed', 'INGLESIDE', 863006)
('Closed', 'CENTRAL', 842397)
('Closed', 'SOUTHERN', 795191)
('Closed', 'TARAVAL', 777268)
('Closed', 'BAYVIEW', 734894)
('Closed', 'PARK', 684583)
('Closed', 'RICHMOND', 676235)
('Closed', None, 387215)
('Closed', 'TENDERLOIN', 361313)
('Open', 'MISSION', 27541)
('Open', 'NORTHERN', 21545)
('Open', 'INGLESIDE', 16854)
('Open', 'CENTRAL', 16058)
('Open', 'TARAVAL', 15936)
('Open', 'SOUTHERN', 15490)
('Open', 'BAYVIEW', 14466)
('Open', 'PARK', 14018)
('Open', 'RICHMOND', 10512)
('Open', 'TENDERLOIN', 5202)
('Open', None, 4162)
('Closed', 'UNASSIGNED', 121)
('Closed', 'Mission', 116)
('Closed', 'Tenderloin', 48)
('Closed', 'Bayview', 19)
('Closed', 'Ingleside', 8)
('Open', 'Tenderloin', 5)
('Open', 'Mission', 4)
('Closed', 'Nob Hill', 3)
('Open', 'UNASSIGNED', 2)
('Open', 'Rincon Hill', 1)
('Open', 'Bayview', 1)
('Closed', 'NO', 1)
('Open', 'South of Market', 1)
